# 02 — Elliptic (base, PyG mirror) EDA

Sanity check of the base dataset (166 features; same graph and labels as Elliptic++ minus the 17 augmented features). Loaded from the raw PyG-mirror CSVs in `data/raw/elliptic/` (headerless: txId, time step, 165 features).

In [1]:
import polars as pl

RESULTS = []

def check(name, expected, actual):
    """Soft verification: record PASS/FAIL, never crash mid-notebook."""
    ok = expected == actual
    RESULTS.append((name, expected, actual, ok))
    print(f"{'PASS' if ok else 'FAIL'}  {name}: expected={expected!r} actual={actual!r}")
    return ok

def summary():
    print("\n=== VERIFICATION SUMMARY ===")
    for name, exp, act, ok in RESULTS:
        print(f"{'PASS' if ok else 'FAIL'}  {name}: expected={exp!r} actual={act!r}")
    n_fail = sum(1 for r in RESULTS if not r[3])
    print(f"{len(RESULTS) - n_fail}/{len(RESULTS)} checks passed")


In [2]:
DATA = "../data/raw/elliptic"
features = pl.read_csv(f"{DATA}/elliptic_txs_features.csv", has_header=False,
                       infer_schema_length=10000)
classes = pl.read_csv(f"{DATA}/elliptic_txs_classes.csv", infer_schema_length=10000)
edges = pl.read_csv(f"{DATA}/elliptic_txs_edgelist.csv", infer_schema_length=10000)
print("features:", features.shape, "classes:", classes.shape, "edges:", edges.shape)

features: (203769, 167) classes: (203769, 2) edges: (234355, 2)


In [3]:
check("tx nodes", 203_769, features.height)
check("directed edges", 234_355, edges.height)
check("feature vector (plan: 166, incl. time step)", 166, features.width - 1)
check("time steps", 49, features[features.columns[1]].n_unique())
lab = classes[classes.columns[-1]].cast(pl.Utf8)
check("illicit", 4_545, (lab == "1").sum())
check("licit", 42_019, (lab == "2").sum())
summary()

PASS  tx nodes: expected=203769 actual=203769
PASS  directed edges: expected=234355 actual=234355
PASS  feature vector (plan: 166, incl. time step): expected=166 actual=166
PASS  time steps: expected=49 actual=49
PASS  illicit: expected=4545 actual=4545
PASS  licit: expected=42019 actual=42019

=== VERIFICATION SUMMARY ===
PASS  tx nodes: expected=203769 actual=203769
PASS  directed edges: expected=234355 actual=234355
PASS  feature vector (plan: 166, incl. time step): expected=166 actual=166
PASS  time steps: expected=49 actual=49
PASS  illicit: expected=4545 actual=4545
PASS  licit: expected=42019 actual=42019
6/6 checks passed
